<a href="https://colab.research.google.com/github/Bensadon/Calculo-Numerico/blob/main/atividade5_david_bensadon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculo Numérico — Atividade 5
## Sistemas lineares e implementação numérica

**Aluno:** david bensadon  
**RA:** 185193  
**Disciplina:** Cálculo Numérico — UNIFESP

---

Este notebook apresenta a resolução completa dos cinco exercícios: formulação matricial, eliminação de Gauss, classificação quanto ao número de soluções, implementação do método em Python e aplicação a um modelo de balanço térmico. As contas manuais são mostradas passo a passo e os resultados numéricos são verificados com `numpy.linalg.solve`.

## 1. Sistemas lineares na forma matricial

A forma matricial de um sistema linear é $A\mathbf{x}=\mathbf{b}$, em que $A$ é a matriz dos coeficientes, $\mathbf{x}$ é o vetor de incógnitas e $\mathbf{b}$ é o vetor independente.

### a) Sistema com duas incógnitas

$$
\begin{cases}
2x+y=7,\\
x-3y=-5.
\end{cases}
\qquad\Longrightarrow\qquad
\underbrace{\begin{bmatrix}2&1\\1&-3\end{bmatrix}}_{A}
\underbrace{\begin{bmatrix}x\\y\end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix}7\\-5\end{bmatrix}}_{\mathbf{b}}.
$$

### b) Sistema com três incógnitas

$$
\begin{cases}
x+2y-z=2,\\
3x-y+2z=9,\\
2x+y+z=7.
\end{cases}
\qquad\Longrightarrow\qquad
\underbrace{\begin{bmatrix}1&2&-1\\3&-1&2\\2&1&1\end{bmatrix}}_{A}
\underbrace{\begin{bmatrix}x\\y\\z\end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix}2\\9\\7\end{bmatrix}}_{\mathbf{b}}.
$$

## 2. Eliminação de Gauss

O sistema é

$$
\begin{cases}
2x+y-z=8,\\
-3x-y+2z=-11,\\
-2x+y+2z=-3.
\end{cases}
$$

Partimos da matriz aumentada e anulamos os elementos abaixo do primeiro pivô. Aplicamos $L_2\leftarrow L_2+\frac{3}{2}L_1$ e $L_3\leftarrow L_3+L_1$:

$$
\left[\begin{array}{ccc|c}
2&1&-1&8\\
-3&-1&2&-11\\
-2&1&2&-3
\end{array}\right]
\longrightarrow
\left[\begin{array}{ccc|c}
2&1&-1&8\\
0&\frac12&\frac12&1\\
0&2&1&5
\end{array}\right].
$$

Para anular o elemento abaixo do segundo pivô, fazemos $L_3\leftarrow L_3-4L_2$:

$$
\left[\begin{array}{ccc|c}
2&1&-1&8\\
0&\frac12&\frac12&1\\
0&0&-1&1
\end{array}\right].
$$

### Substituição regressiva

Da última linha, $-z=1$, portanto $z=-1$. Da segunda linha,

$$\frac12y+\frac12(-1)=1\quad\Longrightarrow\quad y=3.$$

Finalmente, pela primeira linha,

$$2x+3-(-1)=8\quad\Longrightarrow\quad x=2.$$

Logo, a solução é

$$\boxed{(x,y,z)=(2,3,-1)}.$$

## 3. Número de soluções

### a) Infinitas soluções

$$
\left[\begin{array}{cc|c}1&1&3\\2&2&6\end{array}\right]
\xrightarrow{L_2\leftarrow L_2-2L_1}
\left[\begin{array}{cc|c}1&1&3\\0&0&0\end{array}\right].
$$

A segunda equação é múltipla da primeira e a redução produz uma linha nula. Há uma variável livre; tomando $y=t$, obtemos $x=3-t$. Portanto, há **infinitas soluções**:

$$\boxed{(x,y)=(3-t,t),\quad t\in\mathbb{R}}.$$

### b) Nenhuma solução

$$
\left[\begin{array}{cc|c}1&1&3\\2&2&7\end{array}\right]
\xrightarrow{L_2\leftarrow L_2-2L_1}
\left[\begin{array}{cc|c}1&1&3\\0&0&1\end{array}\right].
$$

A última linha representa $0=1$, uma contradição. Logo, o sistema é **inconsistente** e não possui solução.

### c) Solução única

$$
\left[\begin{array}{cc|c}1&1&3\\2&-1&0\end{array}\right]
\xrightarrow{L_2\leftarrow L_2-2L_1}
\left[\begin{array}{cc|c}1&1&3\\0&-3&-6\end{array}\right].
$$

Há pivô nas duas colunas das incógnitas. Da segunda linha, $y=2$; da primeira, $x=1$. Portanto, existe **solução única**:

$$\boxed{(x,y)=(1,2)}.$$

## 4. Implementação da eliminação de Gauss sem pivoteamento

A função abaixo recebe $A$ e $\mathbf{b}$, monta a matriz aumentada, realiza a eliminação abaixo dos pivôs e aplica substituição regressiva. Como o método pedido é **sem pivoteamento**, a função não troca linhas; se encontrar um pivô nulo (ou numericamente muito pequeno), informa que esse caso exigiria troca de linhas ou pivoteamento.

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

In [ ]:
def eliminacao_gauss(A, b, exibir_etapas=True, tolerancia=1e-12):
    """Resolve A x = b por eliminação de Gauss sem pivoteamento.

    Retorna a solução e a matriz aumentada triangular superior.
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1)

    # Verificações básicas das dimensões de entrada.
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A deve ser uma matriz quadrada.")
    if b.size != A.shape[0]:
        raise ValueError("O tamanho de b deve coincidir com o número de linhas de A.")

    n = A.shape[0]
    aumentada = np.column_stack((A.copy(), b.copy()))

    if exibir_etapas:
        print("Matriz aumentada inicial:")
        print(aumentada)

    # Etapa de eliminação: zera os elementos abaixo de cada pivô.
    for k in range(n - 1):
        pivo = aumentada[k, k]
        if abs(pivo) < tolerancia:
            raise ValueError(
                f"Pivô nulo ou muito pequeno na posição ({k + 1}, {k + 1}); "
                "o método sem pivoteamento não pode continuar."
            )

        for i in range(k + 1, n):
            multiplicador = aumentada[i, k] / pivo
            aumentada[i, k:] -= multiplicador * aumentada[k, k:]
            aumentada[i, k] = 0.0  # remove pequeno erro de arredondamento

        if exibir_etapas:
            print(f"\nApós eliminar a coluna {k + 1}:")
            print(aumentada)

    if abs(aumentada[-1, -2]) < tolerancia:
        raise ValueError("Último pivô nulo ou muito pequeno.")

    # Substituição regressiva.
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        soma_conhecida = aumentada[i, i + 1:n] @ x[i + 1:n]
        x[i] = (aumentada[i, -1] - soma_conhecida) / aumentada[i, i]

    if exibir_etapas:
        print("\nSolução calculada:")
        print(x)

    return x, aumentada

### Teste com o sistema do exercício 2

In [ ]:
A_ex2 = np.array([
    [ 2.0,  1.0, -1.0],
    [-3.0, -1.0,  2.0],
    [-2.0,  1.0,  2.0],
])
b_ex2 = np.array([8.0, -11.0, -3.0])

x_gauss, triangular_ex2 = eliminacao_gauss(A_ex2, b_ex2)

Matriz aumentada inicial:
[[  2.   1.  -1.   8.]
 [ -3.  -1.   2. -11.]
 [ -2.   1.   2.  -3.]]

Após eliminar a coluna 1:
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   2.   1.   5. ]]

Após eliminar a coluna 2:
[[ 2.   1.  -1.   8. ]
 [ 0.   0.5  0.5  1. ]
 [ 0.   0.  -1.   1. ]]

Solução calculada:
[ 2.  3. -1.]


In [ ]:
x_numpy = np.linalg.solve(A_ex2, b_ex2)
residuo = A_ex2 @ x_gauss - b_ex2

print("Solução pela implementação: ", x_gauss)
print("Solução por numpy.linalg.solve:", x_numpy)
print("Diferença máxima:            ", np.max(np.abs(x_gauss - x_numpy)))
print("Resíduo A @ x - b:           ", residuo)

assert np.allclose(x_gauss, [2.0, 3.0, -1.0])
assert np.allclose(x_gauss, x_numpy)
assert np.allclose(residuo, 0.0)

Solução pela implementação:  [ 2.  3. -1.]
Solução por numpy.linalg.solve: [ 2.  3. -1.]
Diferença máxima:             5.551115123125783e-16
Resíduo A @ x - b:            [0. 0. 0.]


A implementação produz $[2,3,-1]$, a mesma solução obtida manualmente e por `numpy.linalg.solve` dentro da precisão de ponto flutuante. A diferença máxima observada é da ordem de $10^{-16}$ (erro de arredondamento), e o vetor residual $A\mathbf{x}-\mathbf{b}$ é zero na precisão exibida.

## 5. Modelo de balanço térmico

O modelo é

$$
\begin{bmatrix}
4&-1&0\\
-1&4&-1\\
0&-1&3
\end{bmatrix}
\begin{bmatrix}T_1\\T_2\\T_3\end{bmatrix}
=
\begin{bmatrix}15\\10\\10\end{bmatrix},
$$

ou, em forma de equações,

$$
\begin{cases}
4T_1-T_2=15,\\
-T_1+4T_2-T_3=10,\\
-T_2+3T_3=10.
\end{cases}
$$

### a) Resolução manual

Da primeira equação,

$$T_2=4T_1-15.$$

Da terceira equação,

$$T_3=\frac{10+T_2}{3}=\frac{4T_1-5}{3}.$$

Substituindo essas expressões na segunda equação:

$$
-T_1+4(4T_1-15)-\frac{4T_1-5}{3}=10.
$$

Multiplicando por 3 e reunindo os termos,

$$-3T_1+48T_1-180-4T_1+5=30\quad\Longrightarrow\quad41T_1=205,$$

portanto $T_1=5$. Em seguida,

$$T_2=4(5)-15=5,\qquad T_3=\frac{10+5}{3}=5.$$

Assim,

$$\boxed{T_1=T_2=T_3=5}.$$

### b) Solução em Python

Usaremos tanto a função implementada no exercício 4 quanto `numpy.linalg.solve` para conferir o resultado.

In [ ]:
A_termica = np.array([
    [ 4.0, -1.0,  0.0],
    [-1.0,  4.0, -1.0],
    [ 0.0, -1.0,  3.0],
])
b_termica = np.array([15.0, 10.0, 10.0])

temperaturas_gauss, triangular_termica = eliminacao_gauss(
    A_termica, b_termica, exibir_etapas=True
)
temperaturas_numpy = np.linalg.solve(A_termica, b_termica)

print("\nTemperaturas pela implementação:", temperaturas_gauss)
print("Temperaturas pelo NumPy:       ", temperaturas_numpy)
print("Resíduo:                       ", A_termica @ temperaturas_gauss - b_termica)

assert np.allclose(temperaturas_gauss, [5.0, 5.0, 5.0])
assert np.allclose(temperaturas_gauss, temperaturas_numpy)

Matriz aumentada inicial:
[[ 4. -1.  0. 15.]
 [-1.  4. -1. 10.]
 [ 0. -1.  3. 10.]]

Após eliminar a coluna 1:
[[ 4.   -1.    0.   15.  ]
 [ 0.    3.75 -1.   13.75]
 [ 0.   -1.    3.   10.  ]]

Após eliminar a coluna 2:
[[ 4.       -1.        0.       15.      ]
 [ 0.        3.75     -1.       13.75    ]
 [ 0.        0.        2.733333 13.666667]]

Solução calculada:
[5. 5. 5.]

Temperaturas pela implementação: [5. 5. 5.]
Temperaturas pelo NumPy:        [5. 5. 5.]
Resíduo:                        [0. 0. 0.]


## Conclusão

A eliminação de Gauss transformou os sistemas em formas triangulares, permitindo obter as incógnitas por substituição regressiva. No exercício 2, o resultado foi $(2,3,-1)$. No modelo térmico, as três temperaturas de equilíbrio são iguais a 5. Em ambos os casos, a implementação sem pivoteamento coincidiu com `numpy.linalg.solve` e produziu resíduo nulo.

> **Observação numérica:** embora tenha sido solicitado o método sem pivoteamento, aplicações gerais costumam usar pivoteamento parcial para evitar falhas com pivôs nulos e reduzir erros de arredondamento quando os pivôs são muito pequenos.